# TF × Regulatory Type Analysis

This notebook cross-tabulates transcription factor (TF) motif enrichment with regulatory element type (promoter vs enhancer-like) using HOMER results. The goal is to identify TFs that preferentially associate with promoters or enhancers.

**Analysis steps:**
1. Import required libraries and define file paths
2. Parse motif and annotation results for each peak set
3. Classify peaks as enhancer-like or promoter
4. Cross-tabulate TF occurrence by regulatory type

---

## 1. Import Required Libraries and Define Paths

Import the necessary Python libraries and define the paths to the HOMER motif and annotation files for each peak set.

In [1]:
# Import libraries
import pandas as pd
import os

# Define file paths for each peak set
base_dir = "HOMER/homer_results"
peak_sets = {
    "shared": {
        "motif": os.path.join(base_dir, "shared_peaks/knownMotifs.txt"),
        "annot": os.path.join(base_dir, "shared_peaks/annotated_peaks.txt"),
    },
    "human_specific": {
        "motif": os.path.join(base_dir, "human_specific/knownMotifs.txt"),
        "annot": os.path.join(base_dir, "human_specific/annotated_peaks.txt"),
    },
    "mouse_specific": {
        "motif": os.path.join(base_dir, "mouse_specific/knownMotifs.txt"),
        "annot": os.path.join(base_dir, "mouse_specific/annotated_peaks.txt"),
    },
}
peak_sets

{'shared': {'motif': 'HOMER/homer_results/shared_peaks/knownMotifs.txt',
  'annot': 'HOMER/homer_results/shared_peaks/annotated_peaks.txt'},
 'human_specific': {'motif': 'HOMER/homer_results/human_specific/knownMotifs.txt',
  'annot': 'HOMER/homer_results/human_specific/annotated_peaks.txt'},
 'mouse_specific': {'motif': 'HOMER/homer_results/mouse_specific/knownMotifs.txt',
  'annot': 'HOMER/homer_results/mouse_specific/annotated_peaks.txt'}}

## 2. Parse Motif and Annotation Results

For each peak set, parse the motif and annotation files. Extract the TF motif assignments and regulatory annotation for each peak.

In [2]:
import pandas as pd
import re

def clean_homer_motif(path):
    """
    Robust HOMER knownResults.txt parser (force first N columns, ignore extras).
    """
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    if not lines:
        return pd.DataFrame()
    
    
    colnames = ["Motif Name", "Consensus", "P-value", "Log P-value", "q-value"]
    data_lines = lines[1:]
    
     
    data_lines = [l for l in data_lines if not l.startswith("Motif")]
    
   
    rows = [line.split("\t")[:5] for line in data_lines]
    
    df = pd.DataFrame(rows, columns=colnames)
    
     
    def clean_cell(x):
        x = str(x)
        if re.match(r"^\d+\.\d+\.\d+$", x):
            return ".".join(x.split(".")[:2])
        if "%" in x:
            return x.split("%")[0] + "%"
        return x
        
    df = df.applymap(clean_cell)  
    df["TF"] = df["Motif Name"].astype(str).str.split("/").str[0]
    
    return df

In [3]:
# Use the robust parser for all peak sets
motif_dfs = {k: clean_homer_motif(v['motif']) for k, v in peak_sets.items()}

# Example: show shared motif table
motif_df = motif_dfs['shared']
motif_df

/var/folders/46/2dtjkh8d73xbptjlpgjt_ph00000gn/T/ipykernel_65073/1878088648.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)
/var/folders/46/2dtjkh8d73xbptjlpgjt_ph00000gn/T/ipykernel_65073/1878088648.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)
/var/folders/46/2dtjkh8d73xbptjlpgjt_ph00000gn/T/ipykernel_65073/1878088648.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_cell)


,Motif Name,Consensus,P-value,Log P-value,q-value,TF
0,CTCF(Zf)/CD4+-CTCF-ChIP-Seq(Barski_et_al.)/Homer,AYAGTGCCMYCTRGTGGCCA,1e-783,-1.804e+03,0.0000,CTCF(Zf)
1,BORIS(Zf)/K562-CTCFL-ChIP-Seq(GSE32465)/Homer,CNNBRGCGCCCCCTGSTGGC,1e-413,-9.522e+02,0.0000,BORIS(Zf)
2,NFY(CCAAT)/Promoter/Homer,RGCCAATSRG,1e-364,-8.398e+02,0.0000,NFY(CCAAT)
3,KLF17(Zf)/2cell-Klf17-CutnTag(GSE211845)/Homer,GCYCCGCCCCYH,1e-344,-7.933e+02,0.0000,KLF17(Zf)
4,Sp5(Zf)/mES-Sp5.Flag-ChIP-Seq(GSE72989)/Homer,RGKGGGCGGAGC,1e-332,-7.666e+02,0.0000,Sp5(Zf)
...,...,...,...,...,...,...
467,LRF(Zf)/Erythroblasts-ZBTB7A-ChIP-Seq(GSE74977...,AAGACCCYYN,1e0,0.000e+00,1.0000,LRF(Zf)
468,Snail1(Zf)/LS174T-SNAIL1.HA-ChIP-Seq(GSE127183...,TRCACCTGCY,1e0,0.000e+00,1.0000,Snail1(Zf)
469,Slug(Zf)/Mesoderm-Snai2-ChIP-Seq(GSE61475)/Homer,SNGCACCTGCHS,1e0,0.000e+00,1.0000,Slug(Zf)
470,ZEB1(Zf)/PDAC-ZEB1-ChIP-Seq(GSE64557)/Homer,VCAGGTRDRY,1e0,0.000e+00,1.0000,ZEB1(Zf)


## 3. Cross-tabulate TF Occurrence by Regulatory Type

For each TF, count its occurrence in promoter and enhancer-like peaks. Output a table: TF, promoter, enhancer-like.

In [4]:
# Example: Cross-tabulate for shared peak set
# If you do not have annot_df, just show the top TFs table

def tf_by_reg_type(motif_df, top_n=20):
    top_tfs = motif_df['TF'].head(top_n).tolist()
    return pd.DataFrame({'TF': top_tfs})

# Show top 20 TFs only
# motif_df should be defined already (e.g., motif_df = motif_dfs['shared'])
tf_regtype_df = tf_by_reg_type(motif_df)
tf_regtype_df

,TF
0,CTCF(Zf)
1,BORIS(Zf)
2,NFY(CCAAT)
3,KLF17(Zf)
4,Sp5(Zf)
5,Klf15(Zf)
6,Sp1(Zf)
7,KLF1(Zf)
8,Fli1(ETS)
9,KLF3(Zf)


## 4. Biological Insight: TF Regulatory Preferences

This table highlights TFs that are preferentially associated with promoters or enhancers, providing insight into their regulatory roles.